In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# ✅ Use InfluxDB v2 client (not v3!)
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

print("🔄 InfluxDB Data Export/Import & Cleaning Workflow")
print("=" * 60)

# Configuration
def get_influxdb_config():
    """Get InfluxDB v2 configuration"""
    config = {
        'url': 'http://localhost:8086',
        'token': 'qtab_factor_token_2024_secure',
        'bucket': 'jkp_factors',
        'org': 'qtab'
    }
    return config

config = get_influxdb_config()
print(f"📍 Target: {config['url']}")
print(f"🗄️  Bucket: {config['bucket']}")
print(f"🏢 Organization: {config['org']}")

In [ ]:
def export_complete_jkp_dataset():
    """
    Export the complete JKP dataset from InfluxDB for cleaning
    """
    try:
        # Create InfluxDB v2 client
        client = InfluxDBClient(
            url=config['url'],
            token=config['token'],
            org=config['org'],
            timeout=60000  # Longer timeout for large exports
        )
        
        # Test connection
        health = client.health()
        if health.status != "pass":
            print(f"❌ InfluxDB health check failed: {health.message}")
            return None
        
        print(f"✅ Connected to InfluxDB v{health.version}")
        
        # Query to get ALL data from jkp_characteristics
        query_api = client.query_api()
        
        # Comprehensive Flux query to export everything
        flux_query = f'''
        from(bucket: "{config['bucket']}")
            |> range(start: 1990-01-01T00:00:00Z, stop: 2030-12-31T23:59:59Z)
            |> filter(fn: (r) => r._measurement == "jkp_characteristics")
            |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
            |> drop(columns: ["_start", "_stop", "_measurement"])
        '''
        
        print("🔍 Executing complete data export query...")
        print("⏳ This may take several minutes for large datasets...")
        
        # Execute query and get DataFrame
        result = query_api.query_data_frame(flux_query)
        
        # Handle result (could be list of DataFrames)
        if isinstance(result, list) and len(result) > 0:
            df = result[0]  # Take first DataFrame
            # If multiple DataFrames, concatenate them
            if len(result) > 1:
                df = pd.concat(result, ignore_index=True)
        else:
            df = result
        
        print(f"✅ Export successful!")
        print(f"📊 Records exported: {len(df):,}")
        print(f"📅 Date range: {df['_time'].min()} to {df['_time'].max()}")
        print(f"🏢 Unique companies: {df['gvkey'].nunique() if 'gvkey' in df.columns else 'N/A'}")
        print(f"📈 Columns: {len(df.columns)}")
        
        # Convert _time to datetime and set as index
        df['_time'] = pd.to_datetime(df['_time'])
        
        client.close()
        return df
        
    except Exception as e:
        print(f"❌ Export failed: {e}")
        import traceback
        traceback.print_exc()
        return None

# Execute the export
print("\n🚀 Starting complete dataset export...")
raw_jkp_data = export_complete_jkp_dataset()

if raw_jkp_data is not None:
    print(f"\n📋 Raw data structure:")
    print(f"Shape: {raw_jkp_data.shape}")
    print(f"Columns: {list(raw_jkp_data.columns)}")
    print(f"\nFirst few rows:")
    print(raw_jkp_data.head())
else:
    print("❌ Could not export data - check your InfluxDB connection")

In [ ]:
def comprehensive_data_cleaning(df):
    """
    Comprehensive cleaning and anomaly detection for JKP factor data
    """
    if df is None or df.empty:
        print("❌ No data to clean")
        return None
    
    print("🧹 Starting comprehensive data cleaning...")
    print("=" * 50)
    
    # Make a copy for cleaning
    cleaned_df = df.copy()
    original_shape = cleaned_df.shape
    
    # 1. BASIC DATA TYPE FIXES
    print("🔧 Step 1: Fixing data types...")
    
    # Fix time column
    if '_time' in cleaned_df.columns:
        cleaned_df['_time'] = pd.to_datetime(cleaned_df['_time'])
        cleaned_df = cleaned_df.set_index('_time')
    
    # Fix identifiers
    for col in ['gvkey', 'permno']:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].astype(str).str.strip()
            cleaned_df[col] = cleaned_df[col].replace('nan', np.nan)
    
    # Fix numeric columns
    numeric_cols = []
    for col in cleaned_df.columns:
        if col not in ['gvkey', 'permno', 'size_grp', 'factor_model', 'data_vintage']:
            try:
                cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors='coerce')
                numeric_cols.append(col)
            except:
                continue
    
    print(f"   ✅ Fixed {len(numeric_cols)} numeric columns")
    
    # 2. OUTLIER DETECTION AND TREATMENT
    print("\n🎯 Step 2: Detecting and treating outliers...")
    
    outlier_summary = {}
    for col in numeric_cols:
        if cleaned_df[col].notna().sum() > 100:  # Only for columns with sufficient data
            
            # Calculate outlier thresholds (beyond 3 standard deviations)
            mean_val = cleaned_df[col].mean()
            std_val = cleaned_df[col].std()
            lower_bound = mean_val - 3 * std_val
            upper_bound = mean_val + 3 * std_val
            
            # Also use percentile-based outliers (beyond 1st and 99th percentiles)
            q01 = cleaned_df[col].quantile(0.01)
            q99 = cleaned_df[col].quantile(0.99)
            
            # Count outliers
            extreme_outliers = ((cleaned_df[col] < lower_bound) | (cleaned_df[col] > upper_bound)).sum()
            percentile_outliers = ((cleaned_df[col] < q01) | (cleaned_df[col] > q99)).sum()
            
            outlier_summary[col] = {
                'extreme_outliers': extreme_outliers,
                'percentile_outliers': percentile_outliers,
                'q01': q01,
                'q99': q99,
                'mean': mean_val,
                'std': std_val
            }
            
            # Winsorize at 1st and 99th percentiles (academic standard)
            cleaned_df[col] = cleaned_df[col].clip(lower=q01, upper=q99)
    
    print(f"   ✅ Processed outliers in {len(outlier_summary)} columns")
    
    # 3. MISSING VALUE ANALYSIS
    print("\n🔍 Step 3: Missing value analysis...")
    
    missing_analysis = {}
    for col in cleaned_df.columns:
        missing_count = cleaned_df[col].isna().sum()
        missing_pct = (missing_count / len(cleaned_df)) * 100
        missing_analysis[col] = {
            'count': missing_count,
            'percentage': missing_pct
        }
    
    # Show columns with high missing values
    high_missing = {k: v for k, v in missing_analysis.items() if v['percentage'] > 50}
    if high_missing:
        print(f"   ⚠️  Columns with >50% missing: {list(high_missing.keys())}")
    
    # 4. DUPLICATE DETECTION
    print("\n🔄 Step 4: Duplicate detection...")
    
    # Check for duplicate rows (same time + gvkey)
    if 'gvkey' in cleaned_df.columns:
        duplicates = cleaned_df.reset_index().duplicated(subset=['_time', 'gvkey']).sum()
        if duplicates > 0:
            print(f"   ⚠️  Found {duplicates} duplicate records")
            # Keep the last occurrence (most recent data)
            cleaned_df = cleaned_df.reset_index().drop_duplicates(subset=['_time', 'gvkey'], keep='last').set_index('_time')
        else:
            print(f"   ✅ No duplicates found")
    
    # 5. DATA CONSISTENCY CHECKS
    print("\n🎯 Step 5: Data consistency checks...")
    
    consistency_issues = []
    
    # Check for impossible values
    for col in numeric_cols:
        if 'ratio' in col.lower() or 'me' in col.lower():
            negative_count = (cleaned_df[col] < 0).sum()
            if negative_count > 0:
                consistency_issues.append(f"{col}: {negative_count} negative values")
        
        if 'ret' in col.lower():  # Returns should be reasonable
            extreme_returns = ((cleaned_df[col] < -0.95) | (cleaned_df[col] > 10)).sum()
            if extreme_returns > 0:
                consistency_issues.append(f"{col}: {extreme_returns} extreme returns (>1000% or <-95%)")
    
    if consistency_issues:
        print("   ⚠️  Consistency issues found:")
        for issue in consistency_issues:
            print(f"      • {issue}")
    else:
        print("   ✅ No major consistency issues")
    
    # 6. FINAL SUMMARY
    print(f"\n📊 Cleaning Summary:")
    print("=" * 30)
    print(f"Original shape: {original_shape}")
    print(f"Cleaned shape: {cleaned_df.shape}")
    print(f"Records removed: {original_shape[0] - cleaned_df.shape[0]:,}")
    print(f"Outliers winsorized: {sum(v['percentile_outliers'] for v in outlier_summary.values()):,}")
    print(f"Columns processed: {len(numeric_cols)}")
    
    return cleaned_df, outlier_summary, missing_analysis

# Execute comprehensive cleaning
if raw_jkp_data is not None:
    cleaned_data, outlier_info, missing_info = comprehensive_data_cleaning(raw_jkp_data)
    
    if cleaned_data is not None:
        print("\n✅ Data cleaning completed successfully!")
        
        # Show sample of cleaned data
        print(f"\n📋 Sample of cleaned data:")
        print(cleaned_data.head())
        
        # Save cleaned data locally as backup
        backup_filename = f"jkp_cleaned_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        cleaned_data.to_csv(backup_filename)
        print(f"\n💾 Backup saved: {backup_filename}")
    else:
        print("❌ Data cleaning failed")
else:
    print("❌ No raw data available for cleaning")

In [ ]:
def upload_cleaned_data_to_influxdb(cleaned_df, measurement_name="jkp_characteristics_cleaned"):
    """
    Upload cleaned data back to InfluxDB
    """
    if cleaned_df is None or cleaned_df.empty:
        print("❌ No cleaned data to upload")
        return False
    
    try:
        # Create InfluxDB v2 client
        client = InfluxDBClient(
            url=config['url'],
            token=config['token'],
            org=config['org'],
            timeout=60000
        )
        
        # Test connection
        health = client.health()
        if health.status != "pass":
            print(f"❌ InfluxDB connection failed: {health.message}")
            return False
        
        print(f"🚀 Uploading cleaned data to InfluxDB...")
        print(f"📊 Records to upload: {len(cleaned_df):,}")
        print(f"🎯 Target measurement: {measurement_name}")
        
        # Prepare data for upload
        upload_df = cleaned_df.reset_index().copy()
        
        # Ensure time column is named correctly
        if '_time' in upload_df.columns:
            upload_df['_time'] = pd.to_datetime(upload_df['_time'])
            upload_df = upload_df.set_index('_time')
        
        # Define tag columns
        tag_columns = ['gvkey', 'size_grp', 'factor_model'] if all(col in upload_df.columns for col in ['gvkey', 'size_grp', 'factor_model']) else ['gvkey']
        tag_columns = [col for col in tag_columns if col in upload_df.columns]
        
        # Write data using pandas DataFrame method
        write_api = client.write_api(write_options=SYNCHRONOUS)
        
        # Upload in batches
        batch_size = 5000
        total_batches = (len(upload_df) + batch_size - 1) // batch_size
        
        successful_batches = 0
        for i in range(0, len(upload_df), batch_size):
            batch = upload_df.iloc[i:i + batch_size].copy()
            batch_num = (i // batch_size) + 1
            
            print(f"   📝 Uploading batch {batch_num}/{total_batches} ({len(batch)} records)...")
            
            try:
                # Convert DataFrame to Line Protocol
                points = []
                for idx, row in batch.iterrows():
                    point = Point(measurement_name)
                    
                    # Add timestamp
                    if hasattr(idx, 'to_pydatetime'):
                        point = point.time(idx.to_pydatetime(), WritePrecision.S)
                    
                    # Add tags
                    for tag_col in tag_columns:
                        if tag_col in row and pd.notna(row[tag_col]):
                            point = point.tag(tag_col, str(row[tag_col]))
                    
                    # Add fields (numeric data)
                    for col in row.index:
                        if col not in tag_columns and pd.notna(row[col]):
                            try:
                                if isinstance(row[col], (int, float)):
                                    point = point.field(col, float(row[col]))
                                else:
                                    point = point.field(col, str(row[col]))
                            except:
                                continue
                    
                    points.append(point)
                
                # Write batch
                write_api.write(bucket=config['bucket'], record=points)
                successful_batches += 1
                print(f"      ✅ Batch {batch_num} uploaded successfully")
                
            except Exception as batch_error:
                print(f"      ❌ Failed to upload batch {batch_num}: {batch_error}")
                continue
        
        write_api.close()
        client.close()
        
        print(f"\n📊 Upload Summary:")
        print(f"   ✅ Successful batches: {successful_batches}/{total_batches}")
        print(f"   📈 Records uploaded: ~{successful_batches * batch_size}")
        
        if successful_batches > 0:
            print(f"\n🎉 Cleaned data upload completed!")
            print(f"📊 New measurement: {measurement_name}")
            print(f"🔍 Query using: from(bucket: \"{config['bucket']}\") |> filter(fn: (r) => r._measurement == \"{measurement_name}\")")
            return True
        else:
            print(f"\n❌ Upload failed - no batches were successful")
            return False
        
    except Exception as e:
        print(f"❌ Upload failed: {e}")
        import traceback
        traceback.print_exc()
        return False

# Execute the upload
if 'cleaned_data' in locals() and cleaned_data is not None:
    print("\n🚀 Starting cleaned data upload...")
    upload_success = upload_cleaned_data_to_influxdb(cleaned_data, "jkp_characteristics_cleaned")
    
    if upload_success:
        print("\n✅ Complete workflow finished successfully!")
        print("📋 Available measurements:")
        print("   • jkp_characteristics (original)")
        print("   • jkp_characteristics_cleaned (cleaned)")
    else:
        print("❌ Upload failed")
else:
    print("❌ No cleaned data available for upload")